# Month 1: Data Prep & Baseline
**Objective:** Ingest the CUAD dataset, explore the text scale, and build a robust PDF parser for real-world contracts.

In [ ]:
# Install required libraries for Month 1
!pip install datasets pandas PyMuPDF scikit-learn transformers -q

## 1. Ingesting the CUAD Dataset
We pull the dataset from Hugging Face and convert it to a Pandas DataFrame to inspect the structure and text lengths.

In [ ]:
from datasets import load_dataset
import pandas as pd

# Load the CUAD dataset (SQuAD format)
print("Downloading CUAD dataset...")
dataset = load_dataset("cuad")

# Convert the training split to a DataFrame for easier exploration
train_df = pd.DataFrame(dataset['train'])

print(f"Total training contracts/samples: {len(train_df)}")
display(train_df.head(3))

### 1.1 Supervisor Check: Analyzing Context Windows
Standard BERT models have a maximum sequence length of 512 tokens. Let's look at the character length of these contracts to understand the scale of the chunking problem we will face in Month 2.

In [ ]:
# Calculate character lengths of the 'context' (the contract text)
train_df['context_length'] = train_df['context'].apply(len)

print("--- Contract Text Length Analytics ---")
print(f"Average characters per contract: {train_df['context_length'].mean():.0f}")
print(f"Max characters in a contract:    {train_df['context_length'].max():.0f}")
print(f"Min characters in a contract:    {train_df['context_length'].min():.0f}")

# A typical token is ~4 characters. A 512 token limit is roughly 2,000 characters.
print("\nNotice that the average length far exceeds the ~2000 character limit of standard Transformers.")
print("We will need a sliding window approach for inference later.")

## 2. The PDF Parsing Utility
Real contracts are PDFs. We need a utility that extracts text cleanly, preserving paragraph blocks where possible. We use PyMuPDF (`fitz`).

In [ ]:
import fitz  # PyMuPDF
import re

def parse_contract_pdf(file_path):
    """
    Extracts text from a PDF contract, attempting to preserve layout blocks.
    """
    try:
        # Open the document
        doc = fitz.open(file_path)
        full_text = []
        
        for page_num in range(len(doc)):
            page = doc.load_page(page_num)
            # extract_text("blocks") returns a list of text blocks, 
            # which helps preserve paragraph structure better than raw text.
            blocks = page.get_text("blocks")
            
            for block in blocks:
                # Block structure: (x0, y0, x1, y1, "text", block_no, block_type)
                # Block type 0 is text
                if block[6] == 0:  
                    text = block[4].strip()
                    # Clean up excessive newlines within a single paragraph
                    text = re.sub(r'\n+', ' ', text)
                    if len(text) > 5: # Ignore tiny artifacts/page numbers
                        full_text.append(text)
                        
        return "\n\n".join(full_text)
    
    except Exception as e:
        return f"Error parsing PDF: {str(e)}"

print("Parser utility ready. In practice, you will point this at your uploaded test PDFs.")

## 3. Establishing a Baseline (Zero-Shot)
Before fine-tuning Legal-BERT, we test a standard zero-shot Question Answering pipeline on a small sample to establish a performance floor.

In [ ]:
from transformers import pipeline

# Initialize a standard, non-legal QA pipeline (DistilBERT is fast for a quick notebook test)
print("Loading zero-shot QA baseline...")
qa_pipeline = pipeline("question-answering", model="distilbert-base-cased-distilled-squad")

# Take a small sample contract from our DataFrame
sample_idx = 5
sample_context = train_df['context'].iloc[sample_idx]

# We ask it to find the Governing Law
question = "Under which state or country's laws is this agreement governed?"

print(f"Question: {question}\n")

# Note: Standard pipelines truncate text that is too long.
# For the baseline test, we just pass the first 2500 characters to avoid crashing.
truncated_context = sample_context[:2500]

result = qa_pipeline(question=question, context=truncated_context)
print(f"Baseline Extraction: {result['answer']}")
print(f"Confidence Score: {result['score']:.4f}")
print("\nSupervisor Check: Did it find the correct state, or did it grab random text? Document this baseline.")